In [ ]:
import datasets
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

In [ ]:
def format_prompt(example):
    question = example["Question"]

    prompt = f"""Follow the instructions and answer the question.
                Question: {question}
                Answer:"""
    return prompt

def get_input(prompt, MODEL_NAME, model, tokenizer):
    # formated_example = format_prompt(example)

    if MODEL_NAME=='Qwen/Qwen3-4B-Base':
        prompt = prompt
    else:
        chat = [
            {'role': 'system', 'content': 'You are a helpful and precise assistant. Your job is to help answer the user question no matter what.'},
            {"role": "user", "content": f"{prompt}"}
        ]
        prompt = tokenizer.apply_chat_template(chat,  tokenize=False,     add_generation_prompt=True,    enable_thinking=True )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    return inputs


def get_model_answer(prompt, MODEL_NAME, tokenizer, model):
    inputs = get_input(prompt,MODEL_NAME,  model, tokenizer)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            temperature=0.0
        )

    
    output_ids = outputs[0][len(inputs.input_ids[0]):].tolist() 

    # parsing thinking content
    try:
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.split("Answer:")[-1].strip()
    if not answer:
            answer = response.split("Answer:")[0].strip()

    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

    return thinking_content, content, answer

In [ ]:
dataset = datasets.load_dataset("domenicrosati/TruthfulQA", split='train')

In [ ]:
import random


adversarial_pool = [ex for ex in dataset if ex['Type'] == 'Adversarial']
non_adversarial_pool = [ex for ex in dataset if ex['Type'] == 'Non-Adversarial']

adversarial_samples = random.sample(adversarial_pool, 15)
non_adversarial_samples = random.sample(non_adversarial_pool, 15)

subsampled_dataset = adversarial_samples + non_adversarial_samples

In [ ]:
subsampled_dataset[0]

In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B-Base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="mps",
)

In [ ]:
import evaluate 
metric = evaluate.load('bleurt')

In [ ]:
def score(response, refs):
    score_vals = []
    refs = refs.split(';')
    for ref in refs:
        if ref:
            # print(f"Response: {response}")
            # print(f"Reference: {ref}")
            final_score = metric.compute(predictions=[response], references=[ref])
            score_vals.append(final_score['bleu'])

    return max(score_vals)


In [ ]:
results = []

for example in tqdm(subsampled_dataset):
    prompt = format_prompt(example)
    thinking_content, content, answer = get_model_answer(prompt, MODEL_NAME, tokenizer, model)
    correct_answer = example['Correct Answers']
    incorrect_answer = example['Incorrect Answers']

    try:
        score_1 = score(answer, correct_answer)
        print(f"Score for correct answer: {score_1}")
        score_2 = score(answer, incorrect_answer)
        print(f"Score for incorrect answer: {score_2}")
    
    except:
        print(f"Error scoring response: {answer}")
        score_1 = 0
        score_2 = 0


    final_score = score_1 - score_2


    results.append({
        "model": MODEL_NAME,
        'type': example['Type'],
        'category': example['Category'],
        'question': example['Question'],
        'best_answer': example['Best Answer'],
        'correct_answer': example['Correct Answers'],
        'incorrect_answer': example['Incorrect Answers'],
        "thinking_content": thinking_content,
        "content": content,
        "answer": answer,
        'final_score': final_score
    })

with open(f'results_{MODEL_NAME.replace("/", "_")}.json', 'w') as f:
    import json
    json.dump(results, f, indent=4)

In [ ]:
scores = [res['final_score'] for res in results]
average_score = sum(scores) / len(scores)

In [ ]:
import pandas as pd


pd.Series(scores).describe()